# Iraqi Voice Studio — تدريب RVC v2 على Google Colab

دفتر Colab الرسمي لمنصة **استوديو الصوت العراقي**.

يُنفّذ عبر سكربت التثبيت الرسمي `scripts/colab/install_colab.sh`.

المتغيرات القابلة للتعديل في الخلية الأولى:
- `IRAQI_VOICE_REPO`: مستودع git للمشروع (افتراضيًا المستودع العام؛ اضبطه لنسختك الخاصة).
- `MOUNT_DRIVE`: ربط Google Drive (يُنفَّذ من نواة الجلسة هنا — لا يمكن ربطه من خلية bash).
- `DRIVE_DIR`: مجلد Drive الذي تُحفظ فيه النماذج ونقاط التحقق (افتراضي `RVC_Iraqi`).
- ضع ملفات البيانات الصوتية العراقية (10–60 دقيقة، نظيفة) في `Drive/MyDrive/<DRIVE_DIR>/datasets/<اسم>/`.

**ترتيب التشغيل:** 1 → 2 → 3 → 4، و5 اختيارية. شغّل الخلية الأخيرة (Snapshot) قبل انتهاء الجلسة أو عند أي انقطاع.

**ملاحظات مهمة:**
- نفق Cloudflare يُفعّل المصادقة تلقائيًا: كل نداءات API تعيد 401 حتى تدخل الرمز من تبويب **الإعدادات** في الصفحة (الرمز في `logs/auth.token`).
- عجلات CUDA لـ torch تُثبَّت تلقائيًا حسب درايفر GPU (cu128 للدرايفر 12.8+/13.x)، و`UV_NO_SYNC=1` يُضبط في الجلسة كي لا يعيدها `uv run` إلى CPU.

In [ ]:
# @title 1) الإعداد وربط Drive وفحص GPU (شغّلها أولًا)
IRAQI_VOICE_REPO = ""  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}
DRIVE_DIR = "RVC_Iraqi"  # @param {type:"string"}

import os
if IRAQI_VOICE_REPO:
    os.environ["IRAQI_VOICE_REPO"] = IRAQI_VOICE_REPO
os.environ["DRIVE_DIR"] = DRIVE_DIR

# ربط Drive يجب أن يحدث من نواة الجلسة (هذه الخلية) — من خلية bash يفشل دائمًا
# بخطأ AttributeError: 'NoneType' object has no attribute 'kernel'.
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("تعذّر ربط Drive:", e)
        print("أكمل بدون Drive — نقاط التحقق ستبقى في /content فقط (تُمسح بنهاية الجلسة).")

!nvidia-smi -L || echo "NO GPU"
!python3 --version
print("الخلية التالية: التثبيت الرسمي (سيكتشف Drive المرتبط تلقائيًا).")

In [ ]:
# @title 2) التثبيت الرسمي (uv + عجلات GPU + أصول + استعادة checkpoints إلى /content)
# ينسخ المستودع إلى /content/iraqi-voice-platform ثم يثبّت كل شيء:
# عجلة CUDA المناسبة للدرايفر، الأصول المسبقة، واستعادة runs/models من Drive إلى workspace المحلي.
import os
if os.environ.get("IRAQI_VOICE_REPO"):
    !rm -rf /content/iraqi-voice-platform
    !git clone --depth 1 "$IRAQI_VOICE_REPO" /content/iraqi-voice-platform
%cd /content/iraqi-voice-platform

!bash scripts/colab/install_colab.sh

# تفعيل توجيه الجلسة داخل النواة: مسارات Drive + UV_NO_SYNC الذي يمنع
# `uv run` من إرجاع عجلات torch إلى CPU. كل خلية !uv run لاحقة ترث هذا.
if os.path.exists("colab_wire_env.py"):
    exec(open("colab_wire_env.py").read())
    print(">>> توجيه الجلسة نشط داخل النواة <<<")
    print("RVC_RUNS_DIR =", os.environ.get("RVC_RUNS_DIR", "(محلي)"))
    print("UV_NO_SYNC   =", os.environ.get("UV_NO_SYNC", "0"))
else:
    print("تعذّر إنشاء colab_wire_env.py — تحقق من مخرجات الخلية أعلاه.")
# ---- حزمة الجودة (البحث المكثف — UPGRADE_REPORT_AR.md) ---------------------
# كل مكتبة اختيارية والكود يتحقق قبل استخدامها؛ أي فشل تثبيت لا يوقف شيءًا:
#   speechbrain  -> مقياس تشابه المتحدث ECAPA داخل مقياس الاختيار المركب
#   torchcrepe   -> طريقة F0 "crepe" (تقدير نغمة أقوى على بيانات صاخبة)
#   noisereduce  -> تنظيف الطيف الاختياري قبل التقطيع (--denoise)
!uv pip install --quiet speechbrain torchcrepe noisereduce || echo "(تعذّر تثبيت حزمة الجودة — التدريب يعمل بدونها بخط val_mel)"


In [ ]:
# @title 3) تجهيز البيانات العراقية (تحقق → قص → F0 → ميزات → فهرس)
EXPERIMENT = "iraqi_voice"  # @param {type:"string"}
DATASET_IN_DRIVE = "RVC_Iraqi/datasets/my_voice"  # @param {type:"string"}
SR = "40k"  # @param ["32k", "40k", "48k"]
F0_METHOD = "rmvpe"  # @param ["rmvpe", "fcpe", "pm", "crepe"]
DENOISE = False  # @param {type:"boolean"} (تنظيف طيفي اختياري للبيانات الصاخبة — يتطلب حزمة الجودة)
LOUDNESS_NORM = True  # @param {type:"boolean"} (توحيد جهارة المقاطع — يُنصح به لأي مجموعة بيانات)

# متغيرات الخلية 2 (توجيه Drive وUV_NO_SYNC) موروثة تلقائيًا هنا لأنها في نفس النواة
!uv run rvc-platform data-check "/content/drive/MyDrive/$DATASET_IN_DRIVE"
denoise_flag = "--denoise" if DENOISE else "--no-denoise"
loudness_flag = "--loudness" if LOUDNESS_NORM else "--no-loudness"
!uv run rvc-platform prepare-dataset "/content/drive/MyDrive/$DATASET_IN_DRIVE" "$EXPERIMENT" --sr "$SR" --f0-method "$F0_METHOD" $denoise_flag $loudness_flag

In [ ]:
# @title 4) التدريب (يُستكمل تلقائيًا من آخر Checkpoint عند إعادة التشغيل)
EPOCHS = 200  # @param {type:"integer"}
BATCH = 6  # @param {type:"integer"}
RESUME_FROM = ""  # @param {type:"string"} (اختياري: اسم نقطة تحقق مثل G_48.pth أو latest — اتركه فارغًا لاستكمال الأحدث تلقائيًا)
# ---- وصفة الجودة القصوى (مهيأة افتراضيًا بالفعل — هذه للتحكم الدقيق) ----
# EMA_DECAY: متوسط أوزان المولد المتحرك (0 يعني تعطيل) — يُصدّر ويُقيّم منها النموذج
EMA_DECAY = 0.999  # @param {type:"number"}
# STFT_WEIGHT: خسارة STFT متعددة الدقة (0 تعني الهدف الأصلي فقط)
STFT_WEIGHT = 2.0  # @param {type:"number"}
# USE_MSD: تمييزات متعددة المقاييس الخام (وصفة HiFi-GAN)
USE_MSD = True  # @param {type:"boolean"}
# EARLY_STOP: إيقاف تعاوني بعد N حقبة بلا تحسّن (0 = تعطيل) — يمنع الإفراط بالتدريب
EARLY_STOP = 40  # @param {type:"integer"}
# VAL_SPLIT: نسبة التحقق المحجوز (مقاييس val_mel + تشابه المتحدث + composite)
VAL_SPLIT = 0.05  # @param {type:"number"}

msd_flag = "" if USE_MSD else "--no-msd"
cmd = (f"uv run rvc-platform train start --experiment $EXPERIMENT --epochs {EPOCHS} --batch {BATCH} --sr $SR "
       f"--ema-decay {EMA_DECAY} --stft-weight {STFT_WEIGHT} {msd_flag} "
       f"--early-stop {EARLY_STOP} --val-split {VAL_SPLIT}")
if RESUME_FROM:
    cmd += f" --resume-from {RESUME_FROM}"
print(">>>", cmd)
!{cmd}

# إيقاف تعاوني (يحفظ checkpoint قبل الخروج) من نافذة أخرى أو خلية منفصلة:
#   !uv run rvc-platform train stop

In [ ]:
# @title 4b) (اختياري) تقييم موضوعي لأقوى نقطة تحقق (val_mel + تشابه المتحدث + composite)
# يقيّم أفضل نموذج مُصدَّر على البيانات المحجوزة ويكتب runs/<EXPERIMENT>/eval_report.json
!uv run rvc-platform train evaluate "$EXPERIMENT"

# مقارنة نقطة تدريب أخرى (مثلاً آخر checkpoint بدل الأفضل):
# !uv run rvc-platform train evaluate "$EXPERIMENT" --checkpoint G_48.pth


In [ ]:
# @title 5) (اختياري) واجهة الويب عبر نفق Cloudflare مؤقت
# النفق يُفعّل المصادقة تلقائيًا — الرمز في logs/auth.token:
#   !cat logs/auth.token
# انسخه وأدخله في تبويب الإعدادات داخل الصفحة، وإلا فكل نداءات API تعيد 401.
!uv run rvc-platform web --host 0.0.0.0 --tunnel

In [ ]:
# @title 6) Snapshot إلى Drive قبل انتهاء الجلسة (شغّلها يدويًا عند الانقطاع)
# يوقف التدريب تعاونيًا (مع حفظ checkpoint) ثم ينسخ runs/ و models/ إلى Drive.
# تتم المزامنة عبر ملفات مؤقتة وsnapshot_manifest.json للتحقق بعد الانقطاع.
!uv run rvc-platform train stop
!uv run rvc-platform colab snapshot
!uv run rvc-platform colab status
print(">>> حالّت الجلسة: البيانات محفوظة في Drive. عند جلسة جديدة: شغّل الخليتين 2 و4 فقط — سيُستكمل التدريب تلقائيًا. <<<")